# 02. Sentiment Modeling: Detecting Information Asymmetry

**Context**: In the Kenyan e-commerce landscape, numerical star ratings often mask qualitative dissatisfaction. This notebook builds an NLP pipeline to identify "Information Asymmetry"—cases where products maintain high ratings despite critical feedback hidden in local dialects (Sheng/Swahili).

## 2.1 Load Preprocessed Data

We pull the synthesized NLP dataset created in Phase 1. This dataset already includes slang-mapped tokens and joined product metadata.

In [29]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.data_preprocessing import DataLoader, TextPreprocessor
from src.sentiment_analysis import SentimentModel

# 1. Initialize tools
loader = DataLoader(data_dir="../data/raw/")
text_proc = TextPreprocessor()

# 2. Build the Base NLP Frame
_, prods, revs = loader.load_all()
nlp_df = text_proc.build_nlp_frame(revs, prods)

# 3. Load and Clean Jumia Egypt metadata
egypt_prods = pd.read_csv("../data/raw/jumia_products.csv", encoding="latin1")

# Extract numeric price and rating
egypt_prods["price_numeric"] = egypt_prods["price"].apply(text_proc.parse_price)
egypt_prods["official_rating"] = (
    egypt_prods["avg_rate"].str.extract(r"(\d+\.\d+|\d+)").astype(float)
)
egypt_prods["reviews_count_numeric"] = (
    egypt_prods["reviews_count"].str.extract(r"(\d+)").fillna(0).astype(int)
)

# 4. Attempt Enrichment via SKU Join
enriched_nlp = nlp_df.merge(
    egypt_prods[["id", "official_rating", "reviews_count_numeric", "price_numeric"]],
    left_on="sku",
    right_on="id",
    how="left",
)

# Diagnostic: Verify Join Success
matches = enriched_nlp["official_rating"].notna().sum()
print(f"✅ NLP Frame Ready: {nlp_df.shape[0]} samples")
print(f"🔗 Successfully merged products: {matches}")
if matches == 0:
    print(
        "⚠️ Note: SKU mismatch detected between Kenya Reviews and Egypt Catalog. Proceeding with Cross-Market Inference."
    )

✅ NLP Frame Ready: 92 samples
🔗 Successfully merged products: 0
⚠️ Note: SKU mismatch detected between Kenya Reviews and Egypt Catalog. Proceeding with Cross-Market Inference.


## 2.2 Vectorization & Training

We use TF-IDF to convert tokens into a numerical matrix, capturing nuanced phrases like "feki sana" (very fake) using bigrams.

In [30]:
# Initialize our model class from src/sentiment_analysis.py
from sklearn.model_selection import train_test_split
sm = SentimentModel(max_features=2500)

# 1. Transform text to features
X_tfidf = sm.prepare_features(nlp_df["tokens"])
y = nlp_df["sentiment_target"]

# 2. Train-Test Split (Stratified to maintain sentiment balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Fit the Classifier
sm.train(X_train, y_train)

# 4. Evaluation (This will print the report)
sm.evaluate(X_test, y_test)

sm.save()

--- Sentiment Engine: Detailed Report ---
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         1
         1.0       0.95      1.00      0.97        18

    accuracy                           0.95        19
   macro avg       0.47      0.50      0.49        19
weighted avg       0.90      0.95      0.92        19

☑ Success: Sentiment components saved to ../models/sentiment_rf.joblib and ../models/tfidf_vec.joblib


## 2.3 Cross-Market Inference: Scanning Egypt for Red Flags

The Goal: Since the SKUs don't match, we apply our model to the Egyptian product names. This identifies products that have "High Ratings" but contain "Negative Signal" words (like Used, Refurbished, Damaged) that the model learned were risks in the Kenyan market.

In [31]:
# 1. Sanitize the Catalog: Drop NaNs to avoid Vectorizer crashes
egypt_prods_clean = egypt_prods.dropna(subset=["product_name"]).copy()

# 2. Tokenize product titles
egypt_prods_clean["tokens"] = egypt_prods_clean["product_name"].str.lower().str.split()

# 3. Predict Sentiment based on Learned Patterns
X_egypt = sm.vectorizer.transform(egypt_prods_clean["tokens"])
egypt_prods_clean["predicted_sentiment"] = sm.classifier.predict(X_egypt)

# 4. Detect Information Asymmetry
# Condition: Official rating is High (>= 4.0), but our NLP flags it as Negative (0)
asymmetry_egypt = egypt_prods_clean[
    (egypt_prods_clean["official_rating"] >= 4.0)
    & (egypt_prods_clean["predicted_sentiment"] == 0)
].copy()

asymmetry_egypt = asymmetry_egypt.sort_values(
    by="reviews_count_numeric", ascending=False
)

print(
    f"🕵️ Inference Success: Found {len(asymmetry_egypt)} potential cases in the Egypt Market."
)
asymmetry_egypt[
    ["product_name", "official_rating", "reviews_count_numeric", "price_numeric"]
].head(10)

🕵️ Inference Success: Found 0 potential cases in the Egypt Market.


,product_name,official_rating,reviews_count_numeric,price_numeric


# 2.4 Local Discovery (The "Kenyan Asymmetry")

Finally, we apply the model to our full Kenyan dataset to identify hidden dissatisfaction in reviews that star ratings fail to capture.

In [32]:
# 1. Predict for all Kenyan reviews
X_all = sm.vectorizer.transform(enriched_nlp["tokens"])
enriched_nlp["predicted_sentiment"] = sm.classifier.predict(X_all)

# 2. Filter for Asymmetry
# Note: If 'official_rating' is NaN due to merge issues, these products are skipped
asymmetry_mask = (enriched_nlp["official_rating"] >= 4.0) & (
    enriched_nlp["predicted_sentiment"] == 0
)
asymmetry_df = enriched_nlp[asymmetry_mask]

print(
    f"🕵️ Local Discovery: Found {len(asymmetry_df)} products with hidden qualitative dissatisfaction."
)

# 3. View Results
if not asymmetry_df.empty:
    top_asymmetry = asymmetry_df.sort_values(
        by="reviews_count_numeric", ascending=False
    )
    display(
        top_asymmetry[
            ["product_name", "official_rating", "reviews_count_numeric", "tokens"]
        ].head(10)
    )
else:
    print(
        "No direct SKU matches found to verify local ratings asymmetry. See Inference section (2.3) for cross-market results."
    )

🕵️ Local Discovery: Found 0 products with hidden qualitative dissatisfaction.
No direct SKU matches found to verify local ratings asymmetry. See Inference section (2.3) for cross-market results.
